In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_sales = pd.read_csv(r"C:\Users\ASUS\Desktop\Desktop\Infotact Internship\Project 1 - Omnichannel Retail Sales and Inventory\data\retail_sales_ml_apl.csv")

In [ ]:
df_sales.head()

In [ ]:
print(df_sales.info())
print(df_sales.describe())

In [ ]:
df_sales.isna().sum()

In [ ]:
df_sales['Transaction Date'] = pd.to_datetime(
    df_sales['Transaction Date'],
    dayfirst=True
)

In [ ]:
for col in df_sales.columns.values:
    print(col, ":", len(df_sales[col].unique()))

In [ ]:
#first and last date in the dataset
print(df_sales['Transaction Date'].min())
print(df_sales['Transaction Date'].max())

#total duration covered
start_date = df_sales['Transaction Date'].min()
end_date = df_sales['Transaction Date'].max()
duration = end_date - start_date
print(duration)

#Number of unique dates
print(df_sales['Transaction Date'].nunique())

#number of months covered
print(df_sales['Transaction Date'].dt.month.nunique())

#Extract Start and End Month-Year
print(start_date.strftime('%B %Y'))
print(end_date.strftime('%B %Y'))

In [ ]:
#daily sales
daily_sales = df_sales.groupby('Transaction Date')['Sales Amount'].sum()
daily_sales.head()

daily_sales.plot(figsize=(12,5))

plt.title("Daily Sales Trend")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.show()

In [ ]:
# Extract month and year
df_sales['Year-Month'] = df_sales['Transaction Date'].dt.to_period('M')

# Total sales for each month
monthly_sales = df_sales.groupby('Year-Month')['Sales Amount'].sum()

# Average monthly sales
average_monthly_sales = monthly_sales.mean()

print("Monthly Sales:")
print(monthly_sales)

print("\nAverage Sales Per Month:")
print(average_monthly_sales)

In [ ]:
print(df_sales.groupby('Sales Type')['Sales Amount'].sum())
print(df_sales.groupby('Sales Type')['Sales Amount'].mean())

In [ ]:

# Convert to a datetime object (dayfirst=True handles the DD-MM-YYYY format safely)
df_sales['Transaction Date Parsed'] = pd.to_datetime(df_sales['Transaction Date'], dayfirst=True)

# Filter for rows where Year is 2025 and Month is 6 (June)
df_sales_june2025 = df_sales[
    (df_sales['Transaction Date Parsed'].dt.year == 2025) & 
    (df_sales['Transaction Date Parsed'].dt.month == 6)
].copy()


print(f"June 2025 Sales Rows Extracted: {len(df_sales_june2025)}")

df_sales_june2025

In [ ]:

total_revenue = df_sales['Sales Amount'].sum()
total_cost = df_sales['Cogs'].sum()
total_profit = total_revenue - total_cost
gross_margin_pct = (total_profit / total_revenue) * 100

print("--- COMPANY-WIDE FINANCIAL SUMMARY ---")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Cost of Goods Sold (COGS): ${total_cost:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Gross Profit Margin: {gross_margin_pct:.2f}%\n")

In [ ]:
# Group metrics by store
store_benchmarking = df_sales.groupby('Store').agg(
    Total_Revenue=('Sales Amount', 'sum'),
    Total_Cost=('Cogs', 'sum'),
    Units_Sold=('Qty Sold', 'sum'),
    Transaction_Count=('Number of Transactions', 'sum')
).reset_index()

# Calculate derived metrics per store
store_benchmarking['Total_Profit'] = store_benchmarking['Total_Revenue'] - store_benchmarking['Total_Cost']
store_benchmarking['Gross_Margin_Pct'] = (store_benchmarking['Total_Profit'] / store_benchmarking['Total_Revenue']) * 100

# Rank stores from highest revenue to lowest revenue
store_benchmarking = store_benchmarking.sort_values(by='Total_Revenue', ascending=False).reset_index(drop=True)

# Isolate Top 5 ("Star Stores") and Bottom 5 ("Underperforming Stores")
star_stores = store_benchmarking.head(5)
underperforming_stores = store_benchmarking.tail(5)

print("--- TOP 5 STAR STORES ---")
print(star_stores[['Store', 'Total_Revenue', 'Total_Profit', 'Gross_Margin_Pct', 'Units_Sold']].to_string(index=False))

print("\n--- BOTTOM 5 UNDERPERFORMING STORES ---")
print(underperforming_stores[['Store', 'Total_Revenue', 'Total_Profit', 'Gross_Margin_Pct', 'Units_Sold']].to_string(index=False))


# Saves the sorted metrics for all 40 stores to a flat CSV file
store_benchmarking.to_csv('store_geographical_benchmarking.csv', index=False)